# 🏦 GigScore — Agent 2: XGBoost + SMOTE + Optuna + SHAP
## Behavioural Credit Scoring Engine for India's Informal Economy
**Barclays Hack-O-Hire 2026 · PS4: AI-Powered Alternate Credit Scoring**  
**Team GigScore · Plaksha University · March 28–29, 2026**

> *"Agent 1 built the dataset. Agent 2 builds the brain."*

### What Agent 2 Does
| Step | Action | Output |
|---|---|---|
| 1 | Load Agent 1 outputs | Clean feature matrix |
| 2 | SMOTE on train set only | Balanced training data |
| 3 | Optuna hyperparameter search | Best XGBoost params |
| 4 | Final model training | Trained GigScore model |
| 5 | Full evaluation suite | AUC, KS, F1, confusion matrix |
| 6 | SHAP explainability | Per-applicant feature attribution |
| 7 | GigScore 0-100 mapping | Credit score output |
| 8 | Applicant profiles | Approve / borderline / decline stories |
| 9 | Save model + scores | Handoff to Agent 3 |


## Step 0 — Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
import pickle
import subprocess
warnings.filterwarnings('ignore')

# Install required packages
for pkg in ['xgboost', 'optuna', 'imbalanced-learn', 'shap']:
    subprocess.run(['pip', 'install', pkg, '--break-system-packages', '-q'],
                   capture_output=True)

import xgboost as xgb
from xgboost import XGBClassifier
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
from imblearn.over_sampling import SMOTE
from sklearn.metrics import (
    roc_auc_score, roc_curve, precision_recall_curve,
    confusion_matrix, classification_report,
    f1_score, precision_score, recall_score, average_precision_score
)
from sklearn.calibration import calibration_curve
import shap

plt.rcParams.update({
    'figure.facecolor': '#FAFAFA',
    'axes.facecolor':   '#FFFFFF',
    'axes.spines.top':  False,
    'axes.spines.right':False,
    'axes.grid':        True,
    'grid.alpha':       0.3,
    'font.family':      'DejaVu Sans',
    'font.size':        11,
})

BARCLAYS_BLUE  = '#00AEEF'
BARCLAYS_NAVY  = '#1F4788'
BARCLAYS_EAGLE = '#D4002A'
GREEN          = '#2ECC71'
ORANGE         = '#F39C12'

print("✅ All packages loaded.")
print(f"   XGBoost {xgb.__version__} | Optuna {optuna.__version__}")


## Step 1 — Load Agent 1 Outputs

In [ ]:
# Load the clean feature matrix from Agent 1
X_train = pd.read_csv("X_train.csv", index_col=0)
X_test  = pd.read_csv("X_test.csv",  index_col=0)
y_train = pd.read_csv("y_train.csv", index_col=0).squeeze()
y_test  = pd.read_csv("y_test.csv",  index_col=0).squeeze()

print(f"  Train set : {X_train.shape[0]:>6,} rows × {X_train.shape[1]} features")
print(f"  Test  set : {X_test.shape[0]:>6,} rows × {X_test.shape[1]} features")
print(f"  Train default rate : {y_train.mean():.2%}")
print(f"  Test  default rate : {y_test.mean():.2%}")
print(f"  Class imbalance    : {(y_train==0).sum():,} non-default vs {(y_train==1).sum():,} default")
print(f"  Ratio              : {(y_train==0).sum()/(y_train==1).sum():.1f}:1")
print(f"\n  Features:")

# Categorise features
upi_cols   = [c for c in X_train.columns if c.startswith('upi_')]
syn_cols   = [c for c in X_train.columns if c.startswith('syn_')]
proxy_cols = [c for c in X_train.columns if c in [
    'income_instability_proxy','credit_seeking_intensity','stress_signal',
    'lifestyle_stability','repayment_stretch','credit_utilization_pressure',
    'digital_engagement_proxy']]
cred_cols  = [c for c in X_train.columns if c.startswith('cred_')]
loan_cols  = [c for c in X_train.columns if c.startswith('loan_')]
core_cols  = [c for c in X_train.columns
              if c not in upi_cols+syn_cols+proxy_cols+cred_cols+loan_cols]

print(f"    Home Credit core  : {len(core_cols):>2}")
print(f"    Credit Score      : {len(cred_cols):>2}")
print(f"    Loan Default      : {len(loan_cols):>2}")
print(f"    UPI behavioral    : {len(upi_cols):>2}")
print(f"    Synthetic UPI     : {len(syn_cols):>2}")
print(f"    Proxy behavioral  : {len(proxy_cols):>2}")
print(f"    TOTAL             : {X_train.shape[1]:>2}")


## Step 2 — SMOTE: Fix Class Imbalance

The gig worker cohort has ~11% default rate — a 9:1 class imbalance.
Without correction, XGBoost learns to predict "no default" for everyone and gets 89% accuracy trivially.

**SMOTE** (Synthetic Minority Over-sampling Technique) generates synthetic minority-class examples
in feature space. Applied **on train set only** to prevent data leakage.


In [ ]:
# Apply SMOTE on training set only — never on test set
smote = SMOTE(random_state=42, k_neighbors=5)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print(f"  Before SMOTE: {len(y_train):,} samples | {y_train.mean():.2%} default rate")
print(f"  After  SMOTE: {len(y_train_sm):,} samples | {y_train_sm.mean():.2%} default rate")
print(f"  Synthetic samples added: {len(y_train_sm) - len(y_train):,}")
print(f"  Class balance: {(y_train_sm==0).sum():,} non-default vs {(y_train_sm==1).sum():,} default")

# Visualise class distribution before/after
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
fig.suptitle("Class Distribution: Before vs After SMOTE", fontsize=13, fontweight='bold')

for ax, (y, title) in zip(axes, [(y_train, 'Before SMOTE'), (y_train_sm, 'After SMOTE')]):
    counts = y.value_counts()
    bars = ax.bar(['Non-Default', 'Default'], counts.values,
                  color=[BARCLAYS_NAVY, BARCLAYS_EAGLE], edgecolor='white')
    for bar, val in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                f'{val:,}', ha='center', va='bottom', fontsize=11, fontweight='bold')
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel('Count')

plt.tight_layout()
plt.savefig('chart_smote.png', dpi=150, bbox_inches='tight')
plt.show()
print("  SMOTE chart saved.")


## Step 3 — Optuna Hyperparameter Search

Optuna uses Tree-structured Parzen Estimator (TPE) — a Bayesian optimisation approach that's
much smarter than grid search. It learns from previous trials to focus on promising regions.

**Search space:** n_estimators, max_depth, learning_rate, subsample, colsample_bytree,
min_child_weight, gamma, reg_alpha, reg_lambda

**Target metric:** AUC on a held-out validation fold from the SMOTE-balanced train set.


In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

def objective(trial):
    params = {
        'n_estimators':      trial.suggest_int('n_estimators', 100, 600),
        'max_depth':         trial.suggest_int('max_depth', 3, 8),
        'learning_rate':     trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'subsample':         trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight':  trial.suggest_int('min_child_weight', 1, 10),
        'gamma':             trial.suggest_float('gamma', 0, 5),
        'reg_alpha':         trial.suggest_float('reg_alpha', 0, 2),
        'reg_lambda':        trial.suggest_float('reg_lambda', 0.5, 5),
        'scale_pos_weight':  1,  # balanced by SMOTE
        'random_state':      42,
        'eval_metric':       'auc',
        'verbosity':         0,
        'use_label_encoder': False,
    }
    model = XGBClassifier(**params)
    cv    = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    score = cross_val_score(model, X_train_sm, y_train_sm,
                            cv=cv, scoring='roc_auc', n_jobs=-1)
    return score.mean()

print("  Running Optuna (50 trials) — this takes ~3-5 minutes...")
study = optuna.create_study(direction='maximize',
                             sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=50, show_progress_bar=False)

best_params = study.best_params
best_auc    = study.best_value

print(f"\n  ✅ Optuna complete.")
print(f"  Best CV AUC    : {best_auc:.4f}")
print(f"  Best params    :")
for k, v in best_params.items():
    print(f"    {k:25s}: {v}")


## Step 4 — Train Final GigScore Model

In [ ]:
# Train final model with best Optuna params
final_params = {**best_params,
                'scale_pos_weight': 1,
                'random_state':     42,
                'eval_metric':      'auc',
                'verbosity':        0}

# Remove use_label_encoder if present (deprecated in newer xgboost)
final_params.pop('use_label_encoder', None)

gigscore_model = XGBClassifier(**final_params)
gigscore_model.fit(
    X_train_sm, y_train_sm,
    eval_set=[(X_test, y_test)],
    verbose=False
)

# Predictions
y_prob_train = gigscore_model.predict_proba(X_train)[:, 1]
y_prob_test  = gigscore_model.predict_proba(X_test)[:, 1]
y_pred_test  = (y_prob_test > 0.5).astype(int)

auc_train = roc_auc_score(y_train, y_prob_train)
auc_test  = roc_auc_score(y_test,  y_prob_test)

print(f"  ✅ GigScore model trained.")
print(f"  Train AUC : {auc_train:.4f}")
print(f"  Test  AUC : {auc_test:.4f}")
print(f"  Overfit gap: {auc_train - auc_test:.4f} {'✅ OK' if auc_train - auc_test < 0.03 else '⚠ CHECK'}")
print(f"\n  Classification Report (threshold = 0.5):")
print(classification_report(y_test, y_pred_test,
                             target_names=['Non-Default', 'Default']))


## Step 5 — Full Evaluation Suite

In [ ]:
# ── KS Statistic ─────────────────────────────────────────────────────────
# KS = max separation between default and non-default score distributions
# Industry standard for credit scoring (good model: KS > 0.30)
from scipy.stats import ks_2samp

scores_default    = y_prob_test[y_test == 1]
scores_nondefault = y_prob_test[y_test == 0]
ks_stat, ks_pval  = ks_2samp(scores_nondefault, scores_default)

# ── Gini coefficient ──────────────────────────────────────────────────────
gini = 2 * auc_test - 1

# ── Optimal threshold (F1-maximising) ────────────────────────────────────
prec_arr, rec_arr, thresh_arr = precision_recall_curve(y_test, y_prob_test)
f1_arr     = 2 * prec_arr * rec_arr / (prec_arr + rec_arr + 1e-9)
opt_thresh = thresh_arr[np.argmax(f1_arr[:-1])]
y_pred_opt = (y_prob_test > opt_thresh).astype(int)

f1_opt    = f1_score(y_test, y_pred_opt)
prec_opt  = precision_score(y_test, y_pred_opt)
rec_opt   = recall_score(y_test, y_pred_opt)

# ── Average Precision ─────────────────────────────────────────────────────
avg_prec = average_precision_score(y_test, y_prob_test)

print("╔══════════════════════════════════════════════════════════════════╗")
print("║          GigScore Model — Full Evaluation Suite                  ║")
print("╠══════════════════════════════════════════════════════════════════╣")
print(f"║  AUC (ROC)              : {auc_test:.4f}                          ║")
print(f"║  Gini Coefficient       : {gini:.4f}                          ║")
print(f"║  KS Statistic           : {ks_stat:.4f}  {'✅ GOOD' if ks_stat > 0.30 else '⚠ MODERATE':15s}              ║")
print(f"║  Average Precision (AP) : {avg_prec:.4f}                          ║")
print(f"╠══════════════════════════════════════════════════════════════════╣")
print(f"║  Optimal threshold      : {opt_thresh:.4f}                          ║")
print(f"║  Precision @ opt thresh : {prec_opt:.4f}                          ║")
print(f"║  Recall    @ opt thresh : {rec_opt:.4f}                          ║")
print(f"║  F1        @ opt thresh : {f1_opt:.4f}                          ║")
print("╚══════════════════════════════════════════════════════════════════╝")


In [ ]:
# ── Evaluation Charts ────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.suptitle("GigScore Model — Full Evaluation Suite", fontsize=15, fontweight='bold')

# Chart 1: ROC Curve
ax = axes[0, 0]
fpr, tpr, _ = roc_curve(y_test, y_prob_test)
ax.plot(fpr, tpr, color=BARCLAYS_EAGLE, linewidth=2.5, label=f'GigScore (AUC = {auc_test:.4f})')
ax.fill_between(fpr, tpr, fpr, alpha=0.1, color=BARCLAYS_EAGLE)
ax.plot([0,1],[0,1],'k--', linewidth=1, label='Random')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve', fontweight='bold')
ax.legend(fontsize=10)
ax.text(0.55, 0.15, f'Gini = {gini:.4f}\nKS   = {ks_stat:.4f}',
        transform=ax.transAxes, fontsize=10,
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

# Chart 2: Precision-Recall Curve
ax = axes[0, 1]
ax.plot(rec_arr, prec_arr, color=BARCLAYS_NAVY, linewidth=2.5,
        label=f'GigScore (AP = {avg_prec:.4f})')
ax.axhline(y_test.mean(), color='gray', linestyle='--', linewidth=1.5,
           label=f'Baseline ({y_test.mean():.2%})')
ax.axvline(rec_opt, color=BARCLAYS_EAGLE, linestyle=':', linewidth=1.5,
           label=f'Opt thresh ({opt_thresh:.2f})')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curve', fontweight='bold')
ax.legend(fontsize=9)

# Chart 3: Confusion Matrix
ax = axes[0, 2]
cm = confusion_matrix(y_test, y_pred_opt)
sns.heatmap(cm, annot=True, fmt=',', cmap='Blues', ax=ax,
            xticklabels=['Non-Default', 'Default'],
            yticklabels=['Non-Default', 'Default'],
            cbar=False)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title(f'Confusion Matrix\n(threshold = {opt_thresh:.2f})', fontweight='bold')

# Chart 4: Score Distribution by Target
ax = axes[1, 0]
ax.hist(scores_nondefault, bins=50, alpha=0.7, color=BARCLAYS_NAVY,
        label='Non-Default', density=True)
ax.hist(scores_default,    bins=50, alpha=0.7, color=BARCLAYS_EAGLE,
        label='Default',     density=True)
ax.axvline(opt_thresh, color='black', linestyle='--', linewidth=2,
           label=f'Threshold ({opt_thresh:.2f})')
ax.set_xlabel('Predicted Default Probability')
ax.set_ylabel('Density')
ax.set_title('Score Distribution by Class\nGood separation = good model', fontweight='bold')
ax.legend(fontsize=9)

# Chart 5: KS Chart
ax = axes[1, 1]
sorted_scores = np.sort(y_prob_test)
cum_default    = np.array([np.mean(scores_default    <= s) for s in sorted_scores])
cum_nondefault = np.array([np.mean(scores_nondefault <= s) for s in sorted_scores])
ax.plot(sorted_scores, cum_default,    color=BARCLAYS_EAGLE, linewidth=2, label='Default CDF')
ax.plot(sorted_scores, cum_nondefault, color=BARCLAYS_NAVY,  linewidth=2, label='Non-Default CDF')
ks_idx = np.argmax(np.abs(cum_default - cum_nondefault))
ax.vlines(sorted_scores[ks_idx], cum_nondefault[ks_idx], cum_default[ks_idx],
          color='black', linewidth=2.5, linestyle='--',
          label=f'KS = {ks_stat:.4f}')
ax.set_xlabel('Score')
ax.set_ylabel('Cumulative %')
ax.set_title('KS Chart\nIndustry standard for credit scoring', fontweight='bold')
ax.legend(fontsize=9)

# Chart 6: Calibration Curve
ax = axes[1, 2]
prob_true, prob_pred = calibration_curve(y_test, y_prob_test, n_bins=10)
ax.plot(prob_pred, prob_true, 's-', color=BARCLAYS_EAGLE, linewidth=2, label='GigScore')
ax.plot([0,1],[0,1], 'k--', linewidth=1, label='Perfect calibration')
ax.set_xlabel('Mean Predicted Probability')
ax.set_ylabel('Fraction of Positives')
ax.set_title('Calibration Curve\nHow well-calibrated are the scores?', fontweight='bold')
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('chart_evaluation_suite.png', dpi=150, bbox_inches='tight')
plt.show()
print("  Evaluation suite chart saved.")


## Step 6 — SHAP Explainability

SHAP (SHapley Additive exPlanations) assigns each feature a contribution value for each prediction.
This is what makes GigScore a **product** rather than just a model — lenders can explain every decision.

> *"Why was Ramesh declined? His EXT_SOURCE score is 0.28 (well below median), his credit-income ratio
> is 4.2x (high debt load), and his spending velocity spiked 3 months ago. Likely income shock."*


In [ ]:
# Compute SHAP values on test set
# TreeExplainer is fast and exact for XGBoost
explainer   = shap.TreeExplainer(gigscore_model)
shap_values = explainer.shap_values(X_test)

print(f"  SHAP values computed: {shap_values.shape}")
print(f"  One SHAP value per feature per applicant.")

# Global feature importance via mean |SHAP|
shap_importance = pd.DataFrame({
    'feature':    X_test.columns,
    'mean_shap':  np.abs(shap_values).mean(axis=0)
}).sort_values('mean_shap', ascending=False).reset_index(drop=True)

print(f"\n  Top 15 features by mean |SHAP|:")
print(shap_importance.head(15).to_string(index=False))


In [ ]:
# ── SHAP Charts ──────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(18, 8))
fig.suptitle("GigScore SHAP Explainability\nWhat drives each credit decision?",
             fontsize=14, fontweight='bold')

# Chart 1: Global feature importance (mean |SHAP|)
ax = axes[0]
top15 = shap_importance.head(15)

# colour by feature group
def feature_colour(fname):
    if fname.startswith('upi_'):    return BARCLAYS_BLUE
    if fname.startswith('syn_'):    return GREEN
    if fname.startswith('cred_'):   return ORANGE
    if fname.startswith('loan_'):   return '#9B59B6'
    if fname in ['income_instability_proxy','credit_seeking_intensity',
                 'stress_signal','lifestyle_stability','repayment_stretch',
                 'credit_utilization_pressure','digital_engagement_proxy']:
        return GREEN
    return BARCLAYS_NAVY

colours = [feature_colour(f) for f in top15['feature']]
bars = ax.barh(top15['feature'][::-1], top15['mean_shap'][::-1],
               color=colours[::-1], edgecolor='white')
ax.set_xlabel('Mean |SHAP Value| (impact on model output)', fontsize=11)
ax.set_title('Top 15 Features — Global Importance', fontweight='bold')

# Legend
patches = [
    mpatches.Patch(color=BARCLAYS_NAVY, label='Home Credit core'),
    mpatches.Patch(color=BARCLAYS_BLUE, label='UPI behavioral'),
    mpatches.Patch(color=GREEN,         label='Synthetic/Proxy behavioral'),
    mpatches.Patch(color=ORANGE,        label='Credit Score features'),
    mpatches.Patch(color='#9B59B6',     label='Loan Default features'),
]
ax.legend(handles=patches, fontsize=9, loc='lower right')

# Chart 2: SHAP beeswarm (summary plot) — using matplotlib directly
ax = axes[1]
top10_features = shap_importance.head(10)['feature'].tolist()
top10_shap     = shap_values[:, [X_test.columns.get_loc(f) for f in top10_features]]
top10_vals     = X_test[top10_features].values

# Beeswarm-style scatter
for i, feat in enumerate(top10_features[::-1]):
    feat_idx  = X_test.columns.get_loc(feat)
    sv        = shap_values[:, feat_idx]
    fv        = X_test[feat].values
    fv_norm   = (fv - fv.min()) / (fv.max() - fv.min() + 1e-9)
    jitter    = np.random.normal(0, 0.08, len(sv))
    sc = ax.scatter(sv, np.full_like(sv, i) + jitter,
                    c=fv_norm, cmap='RdYlBu_r',
                    alpha=0.4, s=8, vmin=0, vmax=1)

ax.set_yticks(range(len(top10_features)))
ax.set_yticklabels(top10_features[::-1], fontsize=9)
ax.axvline(0, color='black', linewidth=1)
ax.set_xlabel('SHAP Value (impact on prediction)', fontsize=11)
ax.set_title('SHAP Beeswarm — Top 10 Features\nBlue = low value, Red = high value',
             fontweight='bold')
plt.colorbar(sc, ax=ax, label='Feature value (normalised)')

plt.tight_layout()
plt.savefig('chart_shap_global.png', dpi=150, bbox_inches='tight')
plt.show()
print("  SHAP global chart saved.")


## Step 7 — GigScore 0-100 Mapping

Raw XGBoost outputs probabilities (0-1). We map these to a **GigScore (0-100)** using a
calibrated sigmoid that:
- Inverts direction (high probability of default → low score)
- Stretches the middle range for better differentiation
- Assigns pricing tiers (Prime / Near-Prime / Sub-Prime / Decline)


In [ ]:
def prob_to_gigscore(prob, k=8, midpoint=0.15):
    """
    Map default probability → GigScore 0-100
    k: steepness of sigmoid (higher = more aggressive separation)
    midpoint: probability that maps to GigScore 50
    Inverted: high prob → low score
    """
    # Sigmoid inversion
    score = 100 / (1 + np.exp(k * (prob - midpoint)))
    return np.clip(score, 0, 100)

# Apply to test set
gigscore_test  = prob_to_gigscore(y_prob_test)
gigscore_train = prob_to_gigscore(y_prob_train)

# Tier assignment
def assign_tier(score):
    if score >= 80:   return 'Prime'
    elif score >= 65: return 'Near-Prime'
    elif score >= 50: return 'Sub-Prime'
    else:             return 'Decline'

tiers_test = pd.Series([assign_tier(s) for s in gigscore_test], name='tier')

# Tier stats
tier_order  = ['Prime', 'Near-Prime', 'Sub-Prime', 'Decline']
tier_counts = tiers_test.value_counts()[tier_order]
tier_default= pd.Series({
    t: y_test[tiers_test == t].mean()
    for t in tier_order
    if (tiers_test == t).sum() > 0
})

print("  GigScore Tier Distribution:")
print(f"  {'Tier':<12} {'Count':>7} {'%':>6} {'Default Rate':>13}")
print("  " + "-" * 42)
for t in tier_order:
    if t in tier_counts.index:
        n   = tier_counts[t]
        pct = n / len(tiers_test)
        dr  = tier_default.get(t, 0)
        print(f"  {t:<12} {n:>7,} {pct:>6.1%} {dr:>13.2%}")

print(f"\n  Score statistics:")
print(f"  Mean GigScore : {gigscore_test.mean():.1f}")
print(f"  Median        : {np.median(gigscore_test):.1f}")
print(f"  Std dev       : {gigscore_test.std():.1f}")


In [ ]:
# ── GigScore Distribution Charts ─────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle("GigScore Distribution & Tier Analysis", fontsize=14, fontweight='bold')

# Chart 1: Score distribution with tier colours
ax = axes[0]
n, bins, patches_hist = ax.hist(gigscore_test, bins=50,
                                 edgecolor='white', linewidth=0.5)
for patch, left in zip(patches_hist, bins):
    if   left >= 80: patch.set_facecolor('#166534')
    elif left >= 65: patch.set_facecolor(BARCLAYS_NAVY)
    elif left >= 50: patch.set_facecolor(BARCLAYS_EAGLE)
    else:            patch.set_facecolor('#9CA3AF')

for thresh, label, colour in [(80,'Prime','#166534'),(65,'Near-Prime',BARCLAYS_NAVY),(50,'Sub-Prime',BARCLAYS_EAGLE)]:
    ax.axvline(thresh, color=colour, linestyle='--', linewidth=2, label=label)

total = len(gigscore_test)
text_lines = []
for t, c in [('Prime','#166534'),('Near-Prime',BARCLAYS_NAVY),('Sub-Prime',BARCLAYS_EAGLE),('Decline','#9CA3AF')]:
    if t in tier_counts.index:
        n_t = tier_counts[t]
        text_lines.append(f"{t}: {n_t:,} ({n_t/total:.0%})")
ax.text(0.02, 0.95, '\n'.join(text_lines), transform=ax.transAxes,
        fontsize=9, va='top',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
ax.set_xlabel('GigScore (0-100)')
ax.set_ylabel('Number of Applicants')
ax.set_title('GigScore Distribution\nLoan eligibility tiers', fontweight='bold')
ax.legend(fontsize=9)

# Chart 2: Default rate by tier
ax = axes[1]
tier_colours = {'Prime':'#166534','Near-Prime':BARCLAYS_NAVY,
                'Sub-Prime':BARCLAYS_EAGLE,'Decline':'#9CA3AF'}
tiers_present = [t for t in tier_order if t in tier_default.index]
bars = ax.bar(tiers_present,
              [tier_default[t]*100 for t in tiers_present],
              color=[tier_colours[t] for t in tiers_present],
              edgecolor='white')
for bar, t in zip(bars, tiers_present):
    dr = tier_default[t]
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
            f'{dr:.1%}', ha='center', va='bottom', fontsize=11, fontweight='bold')
ax.axhline(y_test.mean()*100, color='gray', linestyle='--', linewidth=1.5,
           label=f'Overall avg ({y_test.mean():.1%})')
ax.set_ylabel('Default Rate (%)')
ax.set_title('Default Rate by GigScore Tier\nMonotonic decrease = well-calibrated', fontweight='bold')
ax.legend(fontsize=9)

# Chart 3: GigScore vs default probability scatter
ax = axes[2]
sample_idx = np.random.choice(len(gigscore_test), min(2000, len(gigscore_test)), replace=False)
colours_scatter = [BARCLAYS_EAGLE if y == 1 else BARCLAYS_NAVY
                   for y in y_test.iloc[sample_idx]]
ax.scatter(y_prob_test[sample_idx], gigscore_test[sample_idx],
           c=colours_scatter, alpha=0.3, s=8)
ax.set_xlabel('Raw Default Probability')
ax.set_ylabel('GigScore (0-100)')
ax.set_title('Score Mapping: Probability → GigScore\nRed = actual default, Blue = non-default',
             fontweight='bold')
red_p  = mpatches.Patch(color=BARCLAYS_EAGLE, label='Default')
blue_p = mpatches.Patch(color=BARCLAYS_NAVY,  label='Non-Default')
ax.legend(handles=[red_p, blue_p], fontsize=9)

plt.tight_layout()
plt.savefig('chart_gigscore_tiers.png', dpi=150, bbox_inches='tight')
plt.show()
print("  GigScore tier chart saved.")


## Step 8 — Applicant Profiles with SHAP Waterfall

The most powerful demo for judges: show three real applicants from the test set with their
GigScore and the SHAP explanation of every factor that drove it.

**Three profiles:**
1. ✅ **Approved** — high GigScore, strong behavioral signals
2. ⚠️ **Borderline** — medium GigScore, mixed signals
3. ❌ **Declined** — low GigScore, multiple risk flags


In [ ]:
# Find three representative applicants
gigscore_series = pd.Series(gigscore_test, index=X_test.index)

# Approved: top 10% GigScore, actually non-default
approved_mask = (gigscore_series >= gigscore_series.quantile(0.90)) & (y_test == 0)
approved_idx  = gigscore_series[approved_mask].idxmax()

# Borderline: GigScore 45-55
border_mask   = (gigscore_series >= 45) & (gigscore_series <= 55)
border_idx    = gigscore_series[border_mask].sample(1, random_state=42).index[0] if border_mask.sum() > 0 else gigscore_series.iloc[(gigscore_series - 50).abs().argsort()[:1]].index[0]

# Declined: bottom 10% GigScore, actually default
declined_mask = (gigscore_series <= gigscore_series.quantile(0.10)) & (y_test == 1)
declined_idx  = gigscore_series[declined_mask].idxmin() if declined_mask.sum() > 0 else gigscore_series.idxmin()

profiles = {
    '✅ APPROVED':   approved_idx,
    '⚠️ BORDERLINE': border_idx,
    '❌ DECLINED':   declined_idx,
}

print("  Three applicant profiles selected:")
for label, idx in profiles.items():
    gs  = gigscore_series[idx]
    prob= y_prob_test[X_test.index.get_loc(idx)]
    act = y_test[idx]
    tier= assign_tier(gs)
    print(f"  {label:20s} GigScore: {gs:5.1f} | Prob: {prob:.3f} | Tier: {tier} | Actual: {'Default' if act==1 else 'Non-Default'}")


In [ ]:
# ── SHAP Waterfall Charts for each profile ───────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(21, 9))
fig.suptitle("GigScore Applicant Profiles — SHAP Explanation\nWhy did each applicant get their score?",
             fontsize=14, fontweight='bold')

profile_colours = {
    '✅ APPROVED':   '#166534',
    '⚠️ BORDERLINE': ORANGE,
    '❌ DECLINED':   BARCLAYS_EAGLE,
}

for ax, (label, idx) in zip(axes, profiles.items()):
    test_pos  = X_test.index.get_loc(idx)
    sv        = shap_values[test_pos]
    gs        = gigscore_series[idx]
    prob      = y_prob_test[test_pos]
    tier      = assign_tier(gs)
    act       = y_test[idx]

    # Top 10 features by |SHAP| for this applicant
    shap_df   = pd.DataFrame({
        'feature': X_test.columns,
        'shap':    sv,
        'value':   X_test.iloc[test_pos].values
    }).reindex(pd.Series(np.abs(sv)).sort_values(ascending=False).index)
    top10     = shap_df.head(10).iloc[::-1]  # reverse for horizontal bar

    colours_bar = [BARCLAYS_EAGLE if v > 0 else BARCLAYS_NAVY
                   for v in top10['shap']]
    bars = ax.barh(range(len(top10)), top10['shap'],
                   color=colours_bar, edgecolor='white', linewidth=0.5)

    # Feature labels with values
    feat_labels = [f"{f} = {v:.3f}" for f, v in
                   zip(top10['feature'], top10['value'])]
    ax.set_yticks(range(len(top10)))
    ax.set_yticklabels(feat_labels, fontsize=8)
    ax.axvline(0, color='black', linewidth=1)

    colour = profile_colours[label]
    ax.set_title(f"{label}\nGigScore: {gs:.0f}/100 | Tier: {tier}\nProb: {prob:.3f} | Actual: {'Default' if act==1 else 'Non-Default'}",
                 fontweight='bold', color=colour, fontsize=11)
    ax.set_xlabel('SHAP Value (→ increases default risk, ← decreases)')

    # Red/blue patch legend
    red_p  = mpatches.Patch(color=BARCLAYS_EAGLE, label='Increases risk')
    blue_p = mpatches.Patch(color=BARCLAYS_NAVY,  label='Decreases risk')
    ax.legend(handles=[red_p, blue_p], fontsize=8, loc='lower right')

plt.tight_layout()
plt.savefig('chart_shap_profiles.png', dpi=150, bbox_inches='tight')
plt.show()
print("  SHAP profile chart saved.")


## Step 9 — Save Model & Outputs for Agent 3

In [ ]:
# ── Save trained model ───────────────────────────────────────────────────
with open('gigscore_model.pkl', 'wb') as f:
    pickle.dump(gigscore_model, f)

gigscore_model.save_model('gigscore_model.json')

# ── Save scores + SHAP values ─────────────────────────────────────────────
scores_df = pd.DataFrame({
    'applicant_id':      X_test.index,
    'gigscore':          gigscore_test,
    'default_prob':      y_prob_test,
    'tier':              tiers_test.values,
    'actual_default':    y_test.values,
    'predicted_default': y_pred_opt,
})
scores_df.to_csv('GigScore_Predictions.csv', index=False)

# Top 3 SHAP features per applicant (for Agent 3 explanation cards)
top3_shap = []
for i in range(len(X_test)):
    sv      = shap_values[i]
    top_idx = np.abs(sv).argsort()[::-1][:3]
    top3_shap.append({
        'applicant_id': X_test.index[i],
        'gigscore':     gigscore_test[i],
        'feature_1':    X_test.columns[top_idx[0]],
        'shap_1':       sv[top_idx[0]],
        'feature_2':    X_test.columns[top_idx[1]],
        'shap_2':       sv[top_idx[1]],
        'feature_3':    X_test.columns[top_idx[2]],
        'shap_3':       sv[top_idx[2]],
    })
pd.DataFrame(top3_shap).to_csv('GigScore_SHAP_Top3.csv', index=False)

# Save SHAP importance for Agent 3
shap_importance.to_csv('GigScore_SHAP_Importance.csv', index=False)

print("╔══════════════════════════════════════════════════════════════════╗")
print("║         AGENT 2 COMPLETE — SUMMARY                              ║")
print("╠══════════════════════════════════════════════════════════════════╣")
print(f"║  Model             : XGBoost (Optuna-tuned, {final_params['n_estimators']} trees)       ║")
print(f"║  SMOTE             : {len(y_train_sm):,} balanced training samples         ║")
print(f"║  Test AUC          : {auc_test:.4f}                               ║")
print(f"║  Gini Coefficient  : {gini:.4f}                               ║")
print(f"║  KS Statistic      : {ks_stat:.4f} {'✅' if ks_stat > 0.30 else '⚠'}                              ║")
print(f"║  Optimal Threshold : {opt_thresh:.4f}                               ║")
print(f"║  F1 @ opt thresh   : {f1_opt:.4f}                               ║")
print("╠══════════════════════════════════════════════════════════════════╣")
print("║  GigScore Tiers (test set)                                       ║")
for t in tier_order:
    if t in tier_counts.index:
        n_t = tier_counts[t]
        dr  = tier_default.get(t, 0)
        print(f"║    {t:<12}: {n_t:>5,} ({n_t/len(tiers_test):.0%}) | default rate: {dr:.2%}         ║")
print("╠══════════════════════════════════════════════════════════════════╣")
print("║  FILES SAVED                                                     ║")
print("║  📦 gigscore_model.pkl          → trained model                 ║")
print("║  📦 gigscore_model.json         → XGBoost native format         ║")
print("║  📄 GigScore_Predictions.csv    → scores + tiers for all test   ║")
print("║  📄 GigScore_SHAP_Top3.csv      → top 3 SHAP per applicant      ║")
print("║  📄 GigScore_SHAP_Importance.csv→ global feature importance      ║")
print("║  📊 6 evaluation charts + SHAP charts                           ║")
print("╠══════════════════════════════════════════════════════════════════╣")
print("║  NEXT: Agent 3 — GigScore Card UI + Lender Dashboard            ║")
print("╚══════════════════════════════════════════════════════════════════╝")
